# 5) Dashboard StreamView Analytics

## Decisión de negocio

Este dashboard apoya la adquisición y priorización de contenidos. La lectura sigue una secuencia narrativa: primero dimensiona el catálogo, luego identifica oportunidades por género y país, y finalmente muestra títulos y películas que conviene revisar.

> **Nota metodológica:** `popularity` es un índice relativo, no representa reproducciones. Las valoraciones con `vote_average = 0` se consideran no representativas. Presupuesto, ingresos y ROI se analizan únicamente para películas con datos financieros válidos.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML

# Resolver la ruta tanto desde la raíz del proyecto como desde notebooks/.
possible_roots = [Path.cwd(), Path.cwd().parent]
project_root = next((root for root in possible_roots if (root / "data").exists()), Path.cwd())
data_path = project_root / "data" / "streamview_catalogo_limpio.csv"

catalog = pd.read_csv(data_path)
catalog["date_added"] = pd.to_datetime(catalog["date_added"], errors="coerce")
for column in ["release_year", "date_added_year", "popularity", "vote_count", "vote_average", "budget", "revenue", "roi"]:
    catalog[column] = pd.to_numeric(catalog[column], errors="coerce")

catalog["has_rating"] = catalog["vote_average"].gt(0)
catalog["has_financial_data"] = catalog["type"].eq("Movie") & catalog["budget"].gt(0) & catalog["revenue"].gt(0)


def explode_dimension(dataframe, column, output_column):
    dimension = dataframe[["show_id", "type", "title", column]].copy()
    dimension[output_column] = dimension[column].fillna("Sin información").astype(str).str.split(",")
    dimension = dimension.explode(output_column)
    dimension[output_column] = dimension[output_column].str.strip()
    return dimension[~dimension[output_column].isin(["", "Sin información"])]


genres = explode_dimension(catalog, "genres", "genre")
countries = explode_dimension(catalog, "country", "country_name")

print(f"Catálogo cargado: {len(catalog):,} contenidos")
print(f"Películas: {(catalog['type'] == 'Movie').sum():,} | Series: {(catalog['type'] == 'TV Show').sum():,}")
print(f"Fuente: {data_path}")

Catálogo cargado: 31,594 contenidos
Películas: 16,000 | Series: 15,594
Fuente: /home/liquuid/DUOC/Visualización de datos/EV1-VizualicacionDatos/data/streamview_catalogo_limpio.csv


## Panel de exploración

Los filtros siguientes representan una pregunta de adquisición: **qué parte del catálogo quiero comparar y bajo qué criterio de desempeño**. La selección se aplica a KPIs, oportunidades, ranking y evolución.

In [2]:
def options_with_all(values):
    clean_values = sorted(value for value in values if pd.notna(value) and str(value) != "Sin información")
    return ["Todos"] + clean_values


genre_options = options_with_all(genres["genre"].unique())
country_options = options_with_all(countries["country_name"].unique())
year_values = sorted(catalog["release_year"].dropna().astype(int).unique())

content_type = widgets.Dropdown(
    options=["Todos", "Movie", "TV Show"],
    value="Todos",
    description="Tipo:",
    style={"description_width": "initial"},
)
genre_filter = widgets.Dropdown(
    options=genre_options,
    value="Todos",
    description="Género:",
    style={"description_width": "initial"},
)
country_filter = widgets.Dropdown(
    options=country_options,
    value="Todos",
    description="País:",
    style={"description_width": "initial"},
)
year_filter = widgets.SelectionRangeSlider(
    options=year_values,
    index=(0, len(year_values) - 1),
    description="Estreno:",
    continuous_update=False,
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px"),
)
ranking_metric = widgets.Dropdown(
    options=[
        ("Popularidad", "popularity"),
        ("Valoración", "vote_average"),
        ("Votos", "vote_count"),
    ],
    value="popularity",
    description="Ranking:",
    style={"description_width": "initial"},
)
top_n = widgets.IntSlider(
    value=10,
    min=5,
    max=20,
    step=5,
    description="Títulos:",
    continuous_update=False,
    style={"description_width": "initial"},
)

controls = widgets.VBox([
    widgets.HBox([content_type, genre_filter, country_filter]),
    widgets.HBox([year_filter, ranking_metric, top_n]),
])
display(controls)

In [3]:
def format_number(value, decimals=0):
    if pd.isna(value):
        return "Sin datos"
    return f"{value:,.{decimals}f}"


def filter_catalog(selected_type, selected_genre, selected_country, selected_years):
    start_year, end_year = selected_years
    filtered = catalog[catalog["release_year"].between(start_year, end_year, inclusive="both")].copy()

    if selected_type != "Todos":
        filtered = filtered[filtered["type"] == selected_type]
    if selected_genre != "Todos":
        valid_ids = set(genres.loc[genres["genre"] == selected_genre, "show_id"])
        filtered = filtered[filtered["show_id"].isin(valid_ids)]
    if selected_country != "Todos":
        valid_ids = set(countries.loc[countries["country_name"] == selected_country, "show_id"])
        filtered = filtered[filtered["show_id"].isin(valid_ids)]
    return filtered


def render_dashboard(selected_type, selected_genre, selected_country, selected_years, selected_metric, selected_top_n):
    from IPython.display import clear_output

    clear_output(wait=True)
    filtered = filter_catalog(selected_type, selected_genre, selected_country, selected_years)
    if filtered.empty:
        display(HTML("<h3>No hay contenidos para esta combinación de filtros.</h3>"))
        return

    movie_count = int((filtered["type"] == "Movie").sum())
    show_count = int((filtered["type"] == "TV Show").sum())
    rated = filtered.loc[filtered["has_rating"], "vote_average"]
    median_popularity = filtered["popularity"].median()
    average_rating = rated.mean() if not rated.empty else float("nan")

    kpi_style = "display:inline-block; min-width:155px; margin:4px; padding:12px 16px; background:#f4f1ea; border-left:4px solid #e05a47;"
    kpis = "".join([
        f'<div style="{kpi_style}"><small>CONTENIDOS</small><br><b>{len(filtered):,}</b></div>',
        f'<div style="{kpi_style}"><small>PELÍCULAS</small><br><b>{movie_count:,}</b></div>',
        f'<div style="{kpi_style}"><small>SERIES</small><br><b>{show_count:,}</b></div>',
        f'<div style="{kpi_style}"><small>POPULARIDAD MEDIANA</small><br><b>{format_number(median_popularity, 1)}</b></div>',
        f'<div style="{kpi_style}"><small>VALORACIÓN VÁLIDA</small><br><b>{format_number(average_rating, 2)} / 10</b></div>',
    ])
    display(HTML(f'<h2 style="color:#17324d; margin-bottom:4px;">Lectura ejecutiva del segmento seleccionado</h2><div>{kpis}</div>'))

    type_counts = filtered["type"].value_counts().rename_axis("type").reset_index(name="contents")
    fig_type = px.bar(
        type_counts,
        x="type",
        y="contents",
        color="type",
        color_discrete_map={"Movie": "#e05a47", "TV Show": "#1f8a70"},
        labels={"type": "Tipo", "contents": "Contenidos"},
        title="Composición del segmento",
    )
    fig_type.update_layout(showlegend=False, height=360, margin=dict(t=60, l=30, r=20, b=35))

    yearly = filtered.groupby("release_year", as_index=False).agg(contents=("show_id", "nunique"))
    fig_year = px.line(
        yearly,
        x="release_year",
        y="contents",
        markers=True,
        labels={"release_year": "Año de estreno", "contents": "Contenidos"},
        title="Evolución de los estrenos",
    )
    fig_year.update_traces(line_color="#17324d")
    fig_year.update_layout(height=360, margin=dict(t=60, l=30, r=20, b=35))

    filtered_genres = genres[genres["show_id"].isin(filtered["show_id"])]
    genre_summary = (
        filtered_genres.merge(filtered[["show_id", "popularity"]], on="show_id")
        .groupby("genre", as_index=False)
        .agg(contents=("show_id", "nunique"), median_popularity=("popularity", "median"))
        .query("contents >= 25")
        .sort_values("median_popularity", ascending=False)
        .head(10)
    )
    fig_genres = px.bar(
        genre_summary.sort_values("median_popularity"),
        x="median_popularity",
        y="genre",
        orientation="h",
        color="contents",
        color_continuous_scale=["#f4f1ea", "#e05a47"],
        labels={"median_popularity": "Popularidad mediana", "genre": "Género", "contents": "Contenidos"},
        title="Oportunidades: géneros con mayor popularidad típica",
    )
    fig_genres.update_layout(height=420, margin=dict(t=60, l=30, r=20, b=35))

    filtered_countries = countries[countries["show_id"].isin(filtered["show_id"])]
    country_summary = (
        filtered_countries.merge(filtered[["show_id", "popularity"]], on="show_id")
        .groupby("country_name", as_index=False)
        .agg(contents=("show_id", "nunique"), median_popularity=("popularity", "median"))
        .query("contents >= 25")
        .sort_values("median_popularity", ascending=False)
        .head(10)
    )
    fig_countries = px.bar(
        country_summary.sort_values("median_popularity"),
        x="median_popularity",
        y="country_name",
        orientation="h",
        color="contents",
        color_continuous_scale=["#e8f0ed", "#1f8a70"],
        labels={"median_popularity": "Popularidad mediana", "country_name": "País", "contents": "Contenidos"},
        title="Oportunidades: mercados con mayor popularidad típica",
    )
    fig_countries.update_layout(height=420, margin=dict(t=60, l=30, r=20, b=35))

    ranking = (
        filtered[["title", "type", "release_year", "popularity", "vote_average", "vote_count"]]
        .sort_values(selected_metric, ascending=False)
        .head(selected_top_n)
        .copy()
    )
    ranking["metric_value"] = ranking[selected_metric]
    ranking["metric_label"] = selected_metric.replace("_", " ").title()
    fig_ranking = px.bar(
        ranking.sort_values("metric_value"),
        x="metric_value",
        y="title",
        orientation="h",
        color="type",
        color_discrete_map={"Movie": "#e05a47", "TV Show": "#1f8a70"},
        hover_data=["type", "release_year", "vote_average", "vote_count"],
        labels={"metric_value": selected_metric.replace("_", " ").title(), "title": "Título"},
        title=f"Títulos prioritarios por {selected_metric.replace('_', ' ')}",
    )
    fig_ranking.update_layout(height=max(420, selected_top_n * 32), margin=dict(t=60, l=30, r=20, b=35), showlegend=True)

    display(widgets.HBox([widgets.Output(), widgets.Output()]))
    display(fig_type, fig_year)
    display(HTML("<h3 style='color:#17324d;'>Dónde buscar oportunidades</h3><p>Las barras muestran desempeño típico mediante la mediana y conservan el volumen mínimo visible para evitar conclusiones basadas en categorías pequeñas.</p>"))
    display(fig_genres, fig_countries)
    display(HTML("<h3 style='color:#17324d;'>Qué títulos revisar primero</h3>"))
    display(fig_ranking)

    financial = filtered[filtered["has_financial_data"]].copy()
    if not financial.empty:
        financial_top = financial.nlargest(min(10, selected_top_n), "roi")[["title", "roi", "revenue", "budget"]]
        fig_financial = px.bar(
            financial_top.sort_values("roi"),
            x="roi",
            y="title",
            orientation="h",
            color="roi",
            color_continuous_scale=["#e8f0ed", "#1f8a70"],
            labels={"roi": "ROI (ingresos / presupuesto)", "title": "Película"},
            title="Películas con mayor retorno observado",
        )
        fig_financial.update_layout(height=max(360, len(financial_top) * 34), margin=dict(t=60, l=30, r=20, b=35))
        display(HTML("<h3 style='color:#17324d;'>Desempeño financiero de películas</h3><p>Este bloque no compara series y excluye películas sin presupuesto o ingresos válidos.</p>"))
        display(fig_financial)
    else:
        display(HTML("<p><b>Finanzas:</b> el segmento no contiene películas con presupuesto e ingresos válidos.</p>"))

    display(HTML("<p style='color:#555;'><b>Lectura:</b> una categoría atractiva combina presencia suficiente, popularidad típica y títulos concretos para revisar. Estos indicadores sirven para priorizar análisis, no sustituyen la evaluación editorial, contractual o de audiencia.</p>"))


dashboard_output = widgets.interactive_output(
    render_dashboard,
    {
        "selected_type": content_type,
        "selected_genre": genre_filter,
        "selected_country": country_filter,
        "selected_years": year_filter,
        "selected_metric": ranking_metric,
        "selected_top_n": top_n,
    },
)
display(dashboard_output)

Output()

## Criterios de lectura y límites

- Una oportunidad de adquisición no se define solo por tener muchos títulos: se revisan simultáneamente volumen y popularidad mediana.
- Los rankings muestran títulos para revisión humana, no recomendaciones automáticas.
- `popularity` permite comparar visibilidad relativa dentro de la fuente, pero no equivale a reproducciones.
- Las comparaciones financieras están limitadas a películas con presupuesto e ingresos válidos.
- Las categorías pequeñas pueden presentar medianas inestables; por eso los gráficos de oportunidades exigen un mínimo de 25 contenidos.

In [4]:
# Validaciones reproducibles de los supuestos que gobiernan el dashboard.
all_contents = filter_catalog("Todos", "Todos", "Todos", (year_values[0], year_values[-1]))
movies_only = filter_catalog("Movie", "Todos", "Todos", (year_values[0], year_values[-1]))
shows_only = filter_catalog("TV Show", "Todos", "Todos", (year_values[0], year_values[-1]))

assert len(all_contents) == len(catalog)
assert len(movies_only) + len(shows_only) == len(catalog)
assert all_contents["show_id"].is_unique
assert genres["show_id"].isin(catalog["show_id"]).all()
assert countries["show_id"].isin(catalog["show_id"]).all()
assert (catalog.loc[catalog["has_financial_data"], "type"] == "Movie").all()
assert (catalog.loc[catalog["has_financial_data"], ["budget", "revenue"]] > 0).all().all()

print("Validación correcta")
print(f"- Todos los contenidos: {len(all_contents):,}")
print(f"- Películas + series: {len(movies_only):,} + {len(shows_only):,}")
print(f"- Asociaciones de género válidas: {len(genres):,}")
print(f"- Asociaciones de país válidas: {len(countries):,}")
print(f"- Películas con finanzas válidas: {int(catalog['has_financial_data'].sum()):,}")

Validación correcta
- Todos los contenidos: 31,594
- Películas + series: 16,000 + 15,594
- Asociaciones de género válidas: 65,180
- Asociaciones de país válidas: 37,227
- Películas con finanzas válidas: 3,540
